In [32]:
import numpy as np
from rdkit import Chem
import glob

# --- Utilities ---

def read_xyz(filename):
    coords = []
    symbols = []
    with open(filename) as f:
        lines = f.readlines()[2:]  # skip header
        for line in lines:
            parts = line.split()
            if len(parts) < 4:
                continue
            symbols.append(parts[0])
            coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
    return np.array(coords), symbols

def xyz_to_rdkit_mol(symbols):
    mol = Chem.RWMol()
    for s in symbols:
        mol.AddAtom(Chem.Atom(s))
    return mol

def kabsch_align(P, Q):
    P_centered = P - P.mean(axis=0)
    Q_centered = Q - Q.mean(axis=0)
    C = np.dot(P_centered.T, Q_centered)
    V, S, Wt = np.linalg.svd(C)
    d = np.linalg.det(np.dot(Wt.T, V.T))
    D = np.diag([1, 1, d])
    U = np.dot(np.dot(Wt.T, D), V.T)
    P_rot = np.dot(P_centered, U) + Q.mean(axis=0)
    return P_rot

def rmsd(P, Q):
    return np.sqrt(np.mean(np.sum((P - Q)**2, axis=1)))

def distance_matrix(coords):
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    return np.linalg.norm(diff, axis=-1)

def optimize_coords_to_distance_matrix(init_coords, target_dist, lr=0.01, steps=500):
    coords = init_coords.copy()
    for _ in range(steps):
        diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
        dist = np.linalg.norm(diff, axis=-1)
        dist[dist == 0] = 1e-8
        grad = np.zeros_like(coords)
        for i in range(len(coords)):
            grad[i] = np.sum((dist - target_dist)[:, i][:, np.newaxis] * (coords[i] - coords) / dist[:, i][:, np.newaxis], axis=0)
        coords -= lr * grad
    return coords

def write_traj_xyz(filename, frames, symbols):
    """
    frames: list of Nx3 coordinate arrays
    """
    with open(filename, "w") as f:
        for i, coords in enumerate(frames):
            f.write(f"{len(symbols)}\n")
            f.write(f"Frame {i+1}\n")
            for sym, xyz in zip(symbols, coords):
                f.write(f"{sym} {xyz[0]:.6f} {xyz[1]:.6f} {xyz[2]:.6f}\n")

# --- Main iterative averaging procedure ---

# Load all XYZ files
xyz_files = glob.glob("/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/meci/benzene_main_meci/*.xyz")
all_coords = []
all_symbols = []

for file in xyz_files:
    coords, symbols = read_xyz(file)
    all_coords.append(coords)
    all_symbols.append(symbols)

ref_coords = all_coords[0]
ref_symbols = all_symbols[0]
ref_mol = xyz_to_rdkit_mol(ref_symbols)

# Trajectory frames: start with all references
traj_frames = all_coords.copy()

max_iterations = 20
convergence_rmsd = 1e-3  # Å
previous_coords = ref_coords.copy()

for iteration in range(max_iterations):
    aligned_coords_list = []

    for coords, symbols in zip(all_coords, all_symbols):
        mol = xyz_to_rdkit_mol(symbols)
        matches = mol.GetSubstructMatches(ref_mol)
        if not matches:
            raise ValueError("No substructure match found")
        
        # Choose permutation with lowest RMSD
        best_rmsd = np.inf
        best_coords = None
        for match in matches:
            permuted_coords = coords[list(match), :]
            aligned = kabsch_align(permuted_coords, previous_coords)
            current_rmsd = rmsd(aligned, previous_coords)
            if current_rmsd < best_rmsd:
                best_rmsd = current_rmsd
                best_coords = aligned
        aligned_coords_list.append(best_coords)

    # Average distance matrix
    distance_matrices_list = [distance_matrix(c) for c in aligned_coords_list]
    avg_dist_mat = np.mean(np.array(distance_matrices_list), axis=0)

    # Optimize Cartesian coordinates to match averaged distances
    optimized_coords = optimize_coords_to_distance_matrix(previous_coords, avg_dist_mat)

    # Add averaged structure to trajectory
    traj_frames.append(optimized_coords.copy())

    # Check convergence
    delta_rmsd = rmsd(previous_coords, optimized_coords)
    print(f"Iteration {iteration+1}, RMSD change = {delta_rmsd:.6f} Å")
    if delta_rmsd < convergence_rmsd:
        print("Convergence reached.")
        break

    previous_coords = optimized_coords.copy()

# Write trajectory
write_traj_xyz("averaging_traj.xyz", traj_frames, ref_symbols)
print(f"Trajectory written to 'averaging_traj.xyz' with {len(traj_frames)} frames")


Iteration 1, RMSD change = 0.302230 Å
Iteration 2, RMSD change = 0.051289 Å
Iteration 3, RMSD change = 0.013897 Å
Iteration 4, RMSD change = 0.004298 Å
Iteration 5, RMSD change = 0.001449 Å
Iteration 6, RMSD change = 0.000526 Å
Convergence reached.
Trajectory written to 'averaging_traj.xyz' with 9 frames
